# Week 11 — Module 22: Bayesian Optimisation

Realised model accuracy, K-means clustering and function-specific candidate refinement.

## Load observations

In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

# Set thread limits before the K-means analysis.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["LOKY_MAX_CPU_COUNT"] = "1"
warnings.filterwarnings("ignore", category=ConvergenceWarning)

search_roots = [Path.cwd(), *Path.cwd().parents]
relative_locations = [
    Path("10. Module 21/updated_data_week_10"),
    Path("updated_data_week_10"),
    Path("11. Module 22/Data"),
    Path("Data"),
]
expected_before = {1: 19, 2: 19, 3: 24, 4: 39, 5: 29, 6: 29, 7: 39, 8: 49}
expected_after = {i: n + 1 for i, n in expected_before.items()}

def suitable(folder):
    if not all((folder / f"function_{i}.csv").is_file() for i in range(1, 9)):
        return False
    counts = {i: len(pd.read_csv(folder / f"function_{i}.csv")) for i in range(1, 9)}
    return counts in (expected_before, expected_after)

data_folder = next(
    (base / rel for base in search_roots for rel in relative_locations if suitable(base / rel)),
    None,
)
if data_folder is None:
    raise FileNotFoundError("Could not find the eight Week 10 cumulative function CSVs.")

data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}
display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]), "Dimensions": data[i].shape[1] - 1}
    for i in range(1, 9)
]))


,Function,Observations,Dimensions
0,F1,20,2
1,F2,20,2
2,F3,25,3
3,F4,40,4
4,F5,30,4
5,F6,30,5
6,F7,40,6
7,F8,50,8


## Week 10 observations

In [2]:
round11_inputs = {
    1: [0.002495, 0.694432],
    2: [0.702093, 0.186930],
    3: [0.440641, 0.482965, 0.435994],
    4: [0.388750, 0.360865, 0.352431, 0.416723],
    5: [1.000000, 1.000000, 1.000000, 0.969985],
    6: [0.444874, 0.347937, 0.533470, 0.742972, 0.000000],
    7: [0.145227, 0.296570, 0.489754, 0.259986, 0.333711, 0.644995],
    8: [0.000000, 0.221175, 0.182914, 0.081049,
        0.771543, 0.527149, 0.198086, 0.585768]
}

round11_outputs = {
    1: 6.782342707085422e-176,
    2: 0.666152689101189,
    3: -0.010582240224832727,
    4: 0.622949817692231,
    5: 8115.177655264632,
    6: -0.2832282434646809,
    7: 3.024609793718737,
    8: 9.9606106543861
}


## Week 10 results

In [3]:
result_rows = []
for i in range(1, 9):
    df = data[i]
    x_cols, y_col = list(df.columns[:-1]), df.columns[-1]
    point = np.asarray(round11_inputs[i], dtype=float)
    existing = df[x_cols].to_numpy(dtype=float)
    matching = np.flatnonzero(np.all(np.isclose(existing, point, atol=5e-7, rtol=0), axis=1))
    if len(matching):
        saved = float(df.iloc[matching[0]][y_col])
        if not np.isclose(saved, round11_outputs[i], rtol=1e-7, atol=1e-9):
            raise ValueError(f"F{i}: an existing observation has a different output")
        previous_best = float(df.drop(index=df.index[matching[0]])[y_col].max())
    else:
        previous_best = float(df[y_col].max())
    actual = float(round11_outputs[i])
    result_rows.append({"Function": f"F{i}", "Previous best": previous_best,
                        "New output": actual, "Difference": actual - previous_best,
                        "Improved": actual > previous_best})
display(pd.DataFrame(result_rows).round(6))


,Function,Previous best,New output,Difference,Improved
0,F1,0.000000,0.000000,-0.000000,False
1,F2,0.746962,0.666153,-0.080810,False
2,F3,-0.000863,-0.010582,-0.009720,False
3,F4,0.583730,0.622950,0.039220,True
4,F5,8662.482500,8115.177655,-547.304845,False
5,F6,-0.239769,-0.283228,-0.043459,False
6,F7,2.929370,3.024610,0.095240,True
7,F8,9.984981,9.960611,-0.024371,False


## Update observations

In [4]:
for i in range(1, 9):
    df = data[i]
    x_cols, y_col = list(df.columns[:-1]), df.columns[-1]
    point = np.asarray(round11_inputs[i], dtype=float)
    existing = df[x_cols].to_numpy(dtype=float)
    matching = np.flatnonzero(np.all(np.isclose(existing, point, atol=5e-7, rtol=0), axis=1))
    if not len(matching):
        row = dict(zip(x_cols, point))
        row[y_col] = round11_outputs[i]
        data[i] = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]), "Dimensions": data[i].shape[1] - 1}
    for i in range(1, 9)
]))


,Function,Observations,Dimensions
0,F1,20,2
1,F2,20,2
2,F3,25,3
3,F4,40,4
4,F5,30,4
5,F6,30,5
6,F7,40,6
7,F8,50,8


## Realised prediction accuracy

In [5]:
# Predictions recorded for the actual Round 11 submitted points

round11_gp_predictions = {
    1: None,
    2: None,
    3: 0.0115389497,
    4: 0.6499094683,
    5: 8156.3947201423,
    6: -0.2091043932,
    7: 2.9805922474,
    8: 9.9841103723
}

round11_nn_predictions = {
    1: None,
    2: 0.6228064845,
    3: -0.0007285144,
    4: 0.5340044796,
    5: 8144.0434552453,
    6: -0.2306323043,
    7: 2.9539558724,
    8: 10.0852104679
}

accuracy_rows = []
for i in range(1, 9):
    actual = round11_outputs[i]
    gp_pred, nn_pred = round11_gp_predictions[i], round11_nn_predictions[i]
    gp_error = abs(actual - gp_pred) if gp_pred is not None else np.nan
    nn_error = abs(actual - nn_pred) if nn_pred is not None else np.nan
    closer = ("GP" if gp_error < nn_error else "NN" if nn_error < gp_error else "Equal") if np.isfinite(gp_error) and np.isfinite(nn_error) else "—"
    accuracy_rows.append({"Function": f"F{i}", "Actual": actual,
                          "GP prediction": gp_pred, "GP absolute error": gp_error,
                          "NN prediction": nn_pred, "NN absolute error": nn_error,
                          "Closer model": closer})
display(pd.DataFrame(accuracy_rows).round(6))


,Function,Actual,GP prediction,GP absolute error,NN prediction,NN absolute error,Closer model
0,F1,0.000000,NaN,NaN,NaN,NaN,—
1,F2,0.666153,NaN,NaN,0.622806,0.043346,—
2,F3,-0.010582,0.011539,0.022121,-0.000729,0.009854,NN
3,F4,0.622950,0.649909,0.026960,0.534004,0.088945,GP
4,F5,8115.177655,8156.394720,41.217065,8144.043455,28.865800,NN
5,F6,-0.283228,-0.209104,0.074124,-0.230632,0.052596,NN
6,F7,3.024610,2.980592,0.044018,2.953956,0.070654,GP
7,F8,9.960611,9.984110,0.023500,10.085210,0.124600,GP


## Gaussian Process models

In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

gp_models = {}
gp_rows = []
for i in range(1, 9):
    X = data[i].iloc[:, :-1].to_numpy(dtype=float)
    y = data[i].iloc[:, -1].to_numpy(dtype=float)
    kernel = RBF(length_scale=0.2, length_scale_bounds=(1e-5, 1e5))
    gp = GaussianProcessRegressor(
        kernel=kernel, normalize_y=True,
        n_restarts_optimizer=10, random_state=42
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        gp.fit(X, y)
    gp_models[i] = gp
    length_scale = float(gp.kernel_.length_scale)
    gp_rows.append({"Function": f"F{i}", "Observations": len(X),
                    "RBF length scale": length_scale,
                    "Log marginal likelihood": float(gp.log_marginal_likelihood(gp.kernel_.theta)),
                    "At lower bound": length_scale <= 1.000001e-5})
display(pd.DataFrame(gp_rows).round(6))


,Function,Observations,RBF length scale,Log marginal likelihood,At lower bound
0,F1,20,0.000010,-28.378771,True
1,F2,20,0.000010,-28.378771,True
2,F3,25,0.064213,-31.447378,False
3,F4,40,0.271860,-18.841696,False
4,F5,30,0.358698,-16.168186,False
5,F6,30,0.470419,-21.025028,False
6,F7,40,0.337534,-30.458397,False
7,F8,50,1.079281,-15.228370,False


## Neural-network LOOCV

In [7]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

nn_loocv_results = {}
nn_rows = []
for i in range(1, 9):
    X = data[i].iloc[:, :-1].to_numpy(dtype=float)
    y = data[i].iloc[:, -1].to_numpy(dtype=float)
    loo = LeaveOneOut()
    nn_predictions, baseline_predictions, actual_values = [], [], []
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        nn = make_pipeline(
            StandardScaler(),
            MLPRegressor(hidden_layer_sizes=(20,), activation="relu",
                         solver="lbfgs", max_iter=5000, random_state=42)
        )
        nn.fit(X_train, y_train)
        nn_predictions.append(nn.predict(X_test)[0])
        baseline_predictions.append(np.mean(y_train))
        actual_values.append(y_test[0])
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_loocv_results[i] = {"nn_mae": nn_mae, "baseline_mae": baseline_mae,
                            "beats_baseline": beats_baseline}
    nn_rows.append({"Function": f"F{i}", "NN LOOCV MAE": nn_mae,
                    "Baseline MAE": baseline_mae, "Beats baseline": beats_baseline})
display(pd.DataFrame(nn_rows).round(6))


,Function,NN LOOCV MAE,Baseline MAE,Beats baseline
0,F1,0.027752,0.000902,False
1,F2,0.236906,0.239207,True
2,F3,0.046029,0.054884,True
3,F4,1.478298,7.799143,True
4,F5,160.029471,2752.028724,True
5,F6,0.239837,0.562742,True
6,F7,0.473553,0.844545,True
7,F8,0.367202,1.015180,True


## Optimisation progress

In [8]:
progress_results = {}
progress_rows = []
for i in range(1, 9):
    y = data[i].iloc[:, -1].to_numpy(dtype=float)
    running_best = np.maximum.accumulate(y)
    improvement_indices = [0] + [j for j in range(1, len(y)) if running_best[j] > running_best[j-1]]
    best_index = int(np.argmax(y))
    current_best = y[best_index]
    observations_since_best = len(y) - 1 - best_index
    number_improvements = len(improvement_indices) - 1
    recent_window = min(5, len(y))
    recent_improvement = current_best - running_best[-recent_window]
    progress_results[i] = {"current_best": current_best, "best_index": best_index,
        "observations_since_best": observations_since_best, "number_improvements": number_improvements,
        "recent_improvement": recent_improvement, "improvement_indices": improvement_indices}
    progress_rows.append({"Function": f"F{i}", "Current best": current_best,
        "Best observation index": best_index, "Observations since best": observations_since_best,
        "Improvements": number_improvements, "Recent improvement": recent_improvement})
display(pd.DataFrame(progress_rows).round(6))


,Function,Current best,Best observation index,Observations since best,Improvements,Recent improvement
0,F1,0.000000,2,17,2,0.000000
1,F2,0.746962,14,5,2,0.000000
2,F3,-0.000863,22,2,7,0.014636
3,F4,0.622950,39,0,12,0.823380
4,F5,8662.482500,26,3,10,1541.330545
5,F6,-0.239769,28,1,5,0.139970
6,F7,3.024610,39,0,6,0.095240
7,F8,9.984981,48,1,9,0.073732


## K-means cluster selection

In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

kmeans_selection = {}
cluster_selection_rows = []
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak on Windows")
    for i in range(1, 9):
        X = data[i].iloc[:, :-1].to_numpy(dtype=float)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        scores = {}
        for k in range(2, 6):
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
            labels = kmeans.fit_predict(X_scaled)
            scores[k] = silhouette_score(X_scaled, labels)
        best_k = max(scores, key=scores.get)
        kmeans_selection[i] = {"best_k": best_k, "silhouette_score": scores[best_k],
                               "all_scores": scores}
        cluster_selection_rows.append({"Function": f"F{i}", **{f"k={k}": scores[k] for k in range(2,6)},
                                       "Selected k": best_k, "Best silhouette": scores[best_k]})
display(pd.DataFrame(cluster_selection_rows).round(4))


,Function,k=2,k=3,k=4,k=5,Selected k,Best silhouette
0,F1,0.3554,0.3489,0.4066,0.3543,4,0.4066
1,F2,0.5236,0.6462,0.6016,0.5825,3,0.6462
2,F3,0.2987,0.2874,0.2659,0.3073,5,0.3073
3,F4,0.2963,0.2924,0.2850,0.3068,5,0.3068
4,F5,0.4331,0.3764,0.3932,0.4152,2,0.4331
5,F6,0.2483,0.2312,0.2572,0.2286,4,0.2572
6,F7,0.2194,0.2487,0.2740,0.3002,5,0.3002
7,F8,0.1725,0.1780,0.1853,0.1834,4,0.1853


## K-means cluster performance

In [10]:
cluster_results = {}
cluster_summary_rows, cluster_detail_rows = [], []
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak on Windows")
    for i in range(1, 9):
        df = data[i]
        X = df.iloc[:, :-1].to_numpy(dtype=float)
        y = df.iloc[:, -1].to_numpy(dtype=float)
        best_k = kmeans_selection[i]["best_k"]
        silhouette = kmeans_selection[i]["silhouette_score"]
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(X_scaled)
        centroids = scaler.inverse_transform(kmeans.cluster_centers_)
        best_index = np.argmax(y)
        best_cluster = labels[best_index]
        cluster_results[i] = {"labels": labels, "centroids": centroids,
                              "best_cluster": best_cluster, "kmeans": kmeans, "scaler": scaler}
        cluster_summary_rows.append({"Function": f"F{i}", "Selected k": best_k,
                                     "Silhouette": silhouette, "Best cluster": int(best_cluster)})
        for cluster in range(best_k):
            mask = labels == cluster
            cluster_y = y[mask]
            cluster_detail_rows.append({"Function": f"F{i}", "Cluster": cluster,
                "Observations": int(np.sum(mask)), "Mean output": np.mean(cluster_y),
                "Best output": np.max(cluster_y),
                "Centroid": np.array2string(centroids[cluster], precision=6, separator=", ", max_line_width=300)})
display(pd.DataFrame(cluster_summary_rows).round(4))
display(pd.DataFrame(cluster_detail_rows).round(6))


,Function,Selected k,Silhouette,Best cluster
0,F1,4,0.4066,0
1,F2,3,0.6462,2
2,F3,5,0.3073,2
3,F4,5,0.3068,0
4,F5,2,0.4331,1
5,F6,4,0.2572,2
6,F7,5,0.3002,1
7,F8,4,0.1853,0


,Function,Cluster,Observations,Mean output,Best output,Centroid
0,F1,0,7,-0.000515,0.000000,"[0.768764, 0.776421]"
1,F1,1,5,-0.001183,0.000000,"[0.244622, 0.219215]"
2,F1,2,5,0.000000,0.000000,"[0.847663, 0.213129]"
3,F1,3,3,0.000000,0.000000,"[0.107714, 0.818103]"
4,F2,0,5,0.083770,0.244619,"[0.343182, 0.313409]"
5,F2,1,9,0.371778,0.657120,"[0.735324, 0.876753]"
6,F2,2,6,0.591846,0.746962,"[0.70348 , 0.126268]"
7,F3,0,4,-0.078561,-0.036378,"[0.741035, 0.813885, 0.223717]"
8,F3,1,5,-0.056689,-0.015498,"[0.246497, 0.252427, 0.395413]"
9,F3,2,10,-0.046103,-0.000863,"[0.504245, 0.461141, 0.44309 ]"


## Cluster-guided candidates

In [11]:
# STEP 7 - CLUSTER-GUIDED CANDIDATE GENERATION

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

np.random.seed(42)

cluster_candidates = {}
cluster_candidate_rows = []


for i in range(2, 9):

    df = data[i]

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    # Retrieve cluster information from Step 6
    labels = cluster_results[i]["labels"]
    best_cluster = cluster_results[i]["best_cluster"]

    # Observations belonging to the cluster containing the current best
    cluster_X = X[labels == best_cluster]

    # Define the observed range of the promising cluster
    lower = cluster_X.min(axis=0)
    upper = cluster_X.max(axis=0)

    # Generate candidate points inside this region
    candidates = np.random.uniform(
        lower,
        upper,
        size=(50000, X.shape[1])
    )

    # Remove candidates that are extremely close to existing observations
    distances = np.linalg.norm(
        candidates[:, None, :] - X[None, :, :],
        axis=2
    )

    nearest_distance = distances.min(axis=1)

    valid = nearest_distance > 0.01

    candidates_valid = candidates[valid]
    nearest_valid = nearest_distance[valid]

    # GP predictions
    gp = gp_models[i]

    gp_mean, gp_std = gp.predict(
        candidates_valid,
        return_std=True
    )

    # Fit NN using all current observations
    nn = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(20,),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        )
    )

    nn.fit(X, y)
    nn_pred = nn.predict(candidates_valid)

    # Decide whether NN passed current LOOCV validation
    nn_validated = nn_loocv_results[i]["beats_baseline"]

    # Candidate scoring
    #
    # Where NN is validated:
    # average standardised GP and NN predictions.
    #
    # Otherwise:
    # use GP prediction only.

    gp_z = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)

    if nn_validated:

        nn_z = (
            (nn_pred - np.mean(nn_pred))
            / (np.std(nn_pred) + 1e-12)
        )

        score = 0.5 * gp_z + 0.5 * nn_z

    else:
        score = gp_z

    best_idx = np.argmax(score)

    candidate = candidates_valid[best_idx]

    cluster_candidates[i] = candidate
    cluster_candidate_rows.append({"Function": f"F{i}",
        "Candidate": np.array2string(candidate, precision=6, separator=", ", max_line_width=300),
        "GP prediction": gp_mean[best_idx], "GP uncertainty": gp_std[best_idx],
        "NN prediction": nn_pred[best_idx], "NN validated": nn_validated,
        "Nearest distance": nearest_valid[best_idx]})


display(pd.DataFrame(cluster_candidate_rows).round(6))


,Function,Candidate,GP prediction,GP uncertainty,NN prediction,NN validated,Nearest distance
0,F2,"[0.729306, 0.186865]",0.365797,0.260233,0.661354,True,0.027213
1,F3,"[0.428667, 0.516401, 0.455178]",0.037513,0.021492,-0.012296,True,0.039857
2,F4,"[0.397404, 0.357252, 0.361581, 0.434077]",0.722666,0.044381,0.412112,True,0.021532
3,F5,"[0.989719, 0.999174, 0.994916, 0.997164]",8321.188371,3.396656,8326.644042,True,0.011843
4,F6,"[0.410753, 0.388161, 0.491811, 0.803802, 0.010...",-0.228475,0.010458,-0.216792,True,0.050226
5,F7,"[0.152047, 0.265662, 0.514461, 0.311439, 0.322...",3.087322,0.096730,3.025373,True,0.110278
6,F8,"[0.046424, 0.134033, 0.255486, 0.042918, 0.790...",9.933104,0.056333,10.252357,True,0.196623


## Function-specific refinement

In [12]:
# STEP 8 - FUNCTION-SPECIFIC CANDIDATE REFINEMENT

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

np.random.seed(42)

refined_candidates = {}
refinement_rows = []


for i in range(1, 9):

    df = data[i]

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_output = y[best_idx]

    # Fit NN
    nn = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(20,),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        )
    )

    nn.fit(X, y)

    # --------------------------------------------------
    # F1 - MODEL-INDEPENDENT EXPLORATION
    # --------------------------------------------------

    if i == 1:

        candidates = np.random.uniform(
            0,
            1,
            size=(100000, X.shape[1])
        )

        distances = np.linalg.norm(
            candidates[:, None, :] - X[None, :, :],
            axis=2
        )

        nearest = distances.min(axis=1)

        selected = np.argmax(nearest)

        candidate = candidates[selected]


    # --------------------------------------------------
    # F2 - NN-SUPPORTED CLUSTER REFINEMENT
    # --------------------------------------------------

    elif i == 2:

        labels = cluster_results[i]["labels"]
        best_cluster = cluster_results[i]["best_cluster"]

        cluster_X = X[labels == best_cluster]

        lower = cluster_X.min(axis=0)
        upper = cluster_X.max(axis=0)

        candidates = np.random.uniform(
            lower,
            upper,
            size=(100000, X.shape[1])
        )

        distances = np.linalg.norm(
            candidates[:, None, :] - X[None, :, :],
            axis=2
        )

        nearest = distances.min(axis=1)

        valid = nearest > 0.03

        candidates_valid = candidates[valid]
        nearest_valid = nearest[valid]

        nn_pred = nn.predict(candidates_valid)

        selected = np.argmax(nn_pred)

        candidate = candidates_valid[selected]


    # --------------------------------------------------
    # F3, F4, F6, F7, F8 - LOCAL REFINEMENT
    # --------------------------------------------------

    elif i in [3, 4, 6, 7, 8]:

        # Search locally around current best
        radius = 0.10

        candidates = best_point + np.random.uniform(
            -radius,
            radius,
            size=(100000, X.shape[1])
        )

        candidates = np.clip(candidates, 0, 1)

        distances = np.linalg.norm(
            candidates[:, None, :] - X[None, :, :],
            axis=2
        )

        nearest = distances.min(axis=1)

        valid = nearest > 0.03

        candidates_valid = candidates[valid]
        nearest_valid = nearest[valid]

        gp = gp_models[i]

        gp_mean, gp_std = gp.predict(
            candidates_valid,
            return_std=True
        )

        nn_pred = nn.predict(candidates_valid)

        # Standardise predictions
        gp_z = (
            (gp_mean - np.mean(gp_mean))
            / (np.std(gp_mean) + 1e-12)
        )

        nn_z = (
            (nn_pred - np.mean(nn_pred))
            / (np.std(nn_pred) + 1e-12)
        )

        # Function-specific model weighting
        if i == 4:
            # GP was more accurate in latest realised test
            score = 0.7 * gp_z + 0.3 * nn_z

        elif i == 8:
            # GP substantially more accurate recently
            score = 0.8 * gp_z + 0.2 * nn_z

        else:
            # Balanced GP-NN evidence
            score = 0.5 * gp_z + 0.5 * nn_z

        selected = np.argmax(score)

        candidate = candidates_valid[selected]


    # --------------------------------------------------
    # F5 - NOVEL BOUNDARY REFINEMENT
    # --------------------------------------------------

    elif i == 5:

        boundary = np.ones(X.shape[1])

        candidates = boundary + np.random.uniform(
            -0.15,
            0,
            size=(100000, X.shape[1])
        )

        candidates = np.clip(candidates, 0, 1)

        distances = np.linalg.norm(
            candidates[:, None, :] - X[None, :, :],
            axis=2
        )

        nearest = distances.min(axis=1)

        # More novelty than cluster-guided candidate
        valid = nearest > 0.03

        candidates_valid = candidates[valid]
        nearest_valid = nearest[valid]

        gp = gp_models[i]

        gp_mean, gp_std = gp.predict(
            candidates_valid,
            return_std=True
        )

        nn_pred = nn.predict(candidates_valid)

        gp_z = (
            (gp_mean - np.mean(gp_mean))
            / (np.std(gp_mean) + 1e-12)
        )

        nn_z = (
            (nn_pred - np.mean(nn_pred))
            / (np.std(nn_pred) + 1e-12)
        )

        score = 0.5 * gp_z + 0.5 * nn_z

        selected = np.argmax(score)

        candidate = candidates_valid[selected]


    refined_candidates[i] = candidate
    method = ("Space-filling" if i == 1 else "NN cluster" if i == 2
              else "Boundary refinement" if i == 5 else "GP–NN local")
    refinement_rows.append({"Function": f"F{i}", "Method": method,
        "Candidate": np.array2string(candidate, precision=6, separator=", ", max_line_width=300),
        "GP prediction": (np.nan if i in (1, 2) else gp_mean[selected]),
        "GP uncertainty": (np.nan if i in (1, 2) else gp_std[selected]),
        "NN prediction": (np.nan if i == 1 else nn_pred[selected]),
        "Nearest distance": (nearest[selected] if i == 1 else nearest_valid[selected])})
display(pd.DataFrame(refinement_rows).round(6))


,Function,Method,Candidate,GP prediction,GP uncertainty,NN prediction,Nearest distance
0,F1,Space-filling,"[0.249506, 0.999107]",NaN,NaN,NaN,0.246275
1,F2,NN cluster,"[0.665824, 0.186906]",NaN,NaN,0.619287,0.036269
2,F3,GP–NN local,"[0.428706, 0.513778, 0.457382]",0.037818,0.020776,-0.012456,0.038875
3,F4,GP–NN local,"[0.400697, 0.362771, 0.375829, 0.441739]",0.744666,0.118731,0.352589,0.035225
4,F5,Boundary refinement,"[0.967622, 0.993026, 0.99714 , 0.999716]",7899.378880,6.712195,7935.631782,0.033245
5,F6,GP–NN local,"[0.400874, 0.395758, 0.500319, 0.800539, 0.004...",-0.228270,0.015295,-0.211156,0.053264
6,F7,GP–NN local,"[0.176017, 0.276563, 0.482241, 0.296795, 0.315...",3.128395,0.095238,3.074595,0.082579
7,F8,GP–NN local,"[0.077042, 0.136953, 0.185881, 0.16953 , 0.768...",9.998146,0.034109,9.986669,0.109510


## Function 5: targeted boundary check

In [13]:
# STEP 10 - F5 TARGETED BOUNDARY CHECK

i = 5

df = data[i]

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

boundary = np.ones(X.shape[1])

# Generate points close to upper boundary
np.random.seed(42)

candidates = boundary + np.random.uniform(
    -0.08,
    0,
    size=(150000, X.shape[1])
)

candidates = np.clip(candidates, 0, 1)

# Distance from existing observations
distances = np.linalg.norm(
    candidates[:, None, :] - X[None, :, :],
    axis=2
)

nearest = distances.min(axis=1)

# Intermediate novelty window
valid = (nearest >= 0.02) & (nearest <= 0.04)

candidates_valid = candidates[valid]
nearest_valid = nearest[valid]

# GP predictions
gp = gp_models[i]

gp_mean, gp_std = gp.predict(
    candidates_valid,
    return_std=True
)

# NN model
nn = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(20,),
        activation="relu",
        solver="lbfgs",
        max_iter=5000,
        random_state=42
    )
)

nn.fit(X, y)

nn_pred = nn.predict(candidates_valid)

# Standardise predictions
gp_z = (
    (gp_mean - np.mean(gp_mean))
    / (np.std(gp_mean) + 1e-12)
)

nn_z = (
    (nn_pred - np.mean(nn_pred))
    / (np.std(nn_pred) + 1e-12)
)

# Equal GP-NN weighting
score = 0.5 * gp_z + 0.5 * nn_z

selected = np.argmax(score)

candidate = candidates_valid[selected]

boundary_candidate = candidate.copy()
comparison_rows = []
for label, point in [("Cluster-guided", cluster_candidates[5]),
                     ("Novel boundary", refined_candidates[5]),
                     ("Targeted boundary", boundary_candidate)]:
    gp_est, gp_unc = gp.predict(np.asarray(point).reshape(1, -1), return_std=True)
    nn_est = nn.predict(np.asarray(point).reshape(1, -1))[0]
    distance = np.linalg.norm(X - point, axis=1).min()
    comparison_rows.append({"Approach": label,
        "Candidate": np.array2string(point, precision=6, separator=", ", max_line_width=300),
        "GP prediction": float(gp_est[0]), "GP uncertainty": float(gp_unc[0]),
        "NN prediction": float(nn_est), "Nearest distance": float(distance)})
display(pd.DataFrame(comparison_rows).round(6))


,Approach,Candidate,GP prediction,GP uncertainty,NN prediction,Nearest distance
0,Cluster-guided,"[0.989719, 0.999174, 0.994916, 0.997164]",8321.188371,3.396656,8326.644042,0.011843
1,Novel boundary,"[0.967622, 0.993026, 0.99714 , 0.999716]",7899.378880,6.712195,7935.631782,0.033245
2,Targeted boundary,"[0.981966, 0.999701, 0.990442, 0.998795]",8141.386269,4.961240,8155.828807,0.020448


## Final validation and submission

In [14]:
# STEP 11 - FINAL ROUND 12 CANDIDATE VALIDATION

import numpy as np

final_candidates = {
    1: np.array([0.249506, 0.999107]),

    2: np.array([0.729306, 0.186865]),

    3: np.array([
        0.428706, 0.513778, 0.457382
    ]),

    4: np.array([
        0.400697, 0.362771, 0.375829, 0.441739
    ]),

    5: np.array([
        0.981966, 0.999701, 0.990442, 0.998795
    ]),

    6: np.array([
        0.400874, 0.395758, 0.500319,
        0.800539, 0.004733
    ]),

    7: np.array([
        0.176017, 0.276563, 0.482241,
        0.296795, 0.315358, 0.706009
    ]),

    8: np.array([
        0.077042, 0.136953, 0.185881, 0.169530,
        0.768770, 0.510795, 0.209002, 0.691691
    ])
}

validation_rows = []
for i in range(1, 9):
    X = data[i].iloc[:, :-1].to_numpy(dtype=float)
    candidate = final_candidates[i]
    correct_dims = len(candidate) == X.shape[1]
    within_bounds = bool(np.all((candidate >= 0) & (candidate <= 1)))
    distances = np.linalg.norm(X - candidate, axis=1)
    nearest_idx = np.argmin(distances)
    nearest_distance = distances[nearest_idx]
    duplicate = bool(np.any(np.all(np.isclose(X, candidate, rtol=0, atol=1e-8), axis=1)))
    source_candidate = (cluster_candidates[2] if i == 2 else
                        boundary_candidate if i == 5 else refined_candidates[i])
    matches_search = bool(np.all(np.round(candidate, 6) == np.round(source_candidate, 6)))
    validation_rows.append({"Function": f"F{i}",
        "Candidate": np.array2string(candidate, precision=6, separator=", ", max_line_width=300),
        "Dimensions": len(candidate), "Within bounds": within_bounds,
        "Duplicate": duplicate, "Nearest distance": nearest_distance,
        "Matches search": matches_search,
        "Passed": bool(correct_dims and within_bounds and not duplicate and matches_search)})
validation = pd.DataFrame(validation_rows)
display(validation.round(6))

# Verify the reported submissions were produced by the searches in this run.
if not validation["Passed"].all():
    from importlib.metadata import version
    mismatched = validation.loc[~validation["Passed"], "Function"].tolist()
    raise RuntimeError(
        f"Generated points do not match the original submissions for {', '.join(mismatched)}. "
        f"Do not use these points as a reproduced result. "
        f"Environment: scikit-learn {version('scikit-learn')}, SciPy {version('scipy')}."
    )

print("Final submission points (six decimal places):")
for i in range(1, 9):
    print(f"F{i}: " + "-".join(f"{x:.6f}" for x in final_candidates[i]))


,Function,Candidate,Dimensions,Within bounds,Duplicate,Nearest distance,Matches search,Passed
0,F1,"[0.249506, 0.999107]",2,True,False,0.246275,True,True
1,F2,"[0.729306, 0.186865]",2,True,False,0.027213,True,True
2,F3,"[0.428706, 0.513778, 0.457382]",3,True,False,0.038875,True,True
3,F4,"[0.400697, 0.362771, 0.375829, 0.441739]",4,True,False,0.035225,True,True
4,F5,"[0.981966, 0.999701, 0.990442, 0.998795]",4,True,False,0.020448,True,True
5,F6,"[0.400874, 0.395758, 0.500319, 0.800539, 0.004...",5,True,False,0.053264,True,True
6,F7,"[0.176017, 0.276563, 0.482241, 0.296795, 0.315...",6,True,False,0.082578,True,True
7,F8,"[0.077042, 0.136953, 0.185881, 0.16953 , 0.768...",8,True,False,0.109510,True,True


Final submission points (six decimal places):
F1: 0.249506-0.999107
F2: 0.729306-0.186865
F3: 0.428706-0.513778-0.457382
F4: 0.400697-0.362771-0.375829-0.441739
F5: 0.981966-0.999701-0.990442-0.998795
F6: 0.400874-0.395758-0.500319-0.800539-0.004733
F7: 0.176017-0.276563-0.482241-0.296795-0.315358-0.706009
F8: 0.077042-0.136953-0.185881-0.169530-0.768770-0.510795-0.209002-0.691691
